# T2.2 — Semantic Mapping Upload to DBRepo

Uploads all semantic mappings from `docs/semantic_mapping.csv` to DBRepo via REST API.

**Owner:** Person B | **Task:** T2.2 — Semantic Mapping | **Dataset:** Hohe Warte Vienna Weather

> **Requires:** TU Wien VPN active, and the group database already created in DBRepo (T2.1 complete).

## Step 0 — Install the official DBRepo Python library

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dbrepo', '--quiet'])
print('dbrepo library ready')

# Verify the exact method signature we will use
from dbrepo.RestClient import RestClient
import inspect
print()
print('update_table_column signature:')
print(inspect.signature(RestClient.update_table_column))
print(inspect.getdoc(RestClient.update_table_column))



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


dbrepo library ready

update_table_column signature:
(self, database_id: str, table_id: str, column_id: str, concept_uri: str = None, unit_uri: str = None) -> dbrepo.api.dto.Column
Update semantic information of a table column by given database id and table id and column id.

:param database_id: The database id.
:param table_id: The table id.
:param column_id: The column id.
:param concept_uri: The concept URI. Optional.
:param unit_uri: The unit URI. Optional.

:returns: The column, if successful.

:raises MalformedError: If the payload is rejected by the service.
:raises ForbiddenError: If something went wrong with the authorization.
:raises NotExistsError: If the accept header is neither application/json nor application/ld+json.
:raises ServiceConnectionError: If something went wrong with the connection to the search service.
:raises ServiceError: If something went wrong with obtaining the information in the search service.
:raises ResponseCodeError: If something went wrong with the

## Step 1 — Configuration

> Password is loaded from the `DBREPO_PASSWORD` environment variable, or prompted at runtime.
> **Never hardcode or commit your real password.**

In [4]:
import os

ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"
USERNAME    = "azra1558"
PASSWORD    = "Katalizator1558!"

candidates = [
    "../docs/semantic_mapping.csv",
    "docs/semantic_mapping.csv",
    "semantic_mapping.csv"
]

MAPPING_FILE = next((c for c in candidates if os.path.exists(c)), None)
if MAPPING_FILE is None:
    searched = "\n".join(f"  {os.path.abspath(c)}" for c in candidates)
    raise FileNotFoundError(
        f"semantic_mapping.csv not found. Searched:\n{searched}\n"
        f"Current working directory: {os.getcwd()}"
    )

print(f"Mapping file found: {os.path.abspath(MAPPING_FILE)}")

Mapping file found: /workspaces/Vienna-Weather-Wet-Month-Prediction/docs/semantic_mapping.csv


## Step 2 — Connect to DBRepo and fetch database structure

The API requires **table UUID** and **column UUID** — not names. We fetch them here.

In [5]:
from dbrepo.RestClient import RestClient

# RestClient handles authentication internally
client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

db = client.get_database(database_id=DATABASE_ID)
print(f"Connected to database: {db.name}")

# db.tables is Optional — guard against None
tables = db.tables or []
print(f"Tables found: {[t.name for t in tables]}")
if not tables:
    raise RuntimeError("No tables found — check DATABASE_ID or ensure T2.1 is complete")


Connected to database: vienna_weather_wet_months
Tables found: ['weather_measurement', 'time_dimension', 'station']


## Step 3 — Build name-to-ID lookup maps

In [6]:
table_id_map  = {}  # table_name -> table UUID
column_id_map = {}  # (table_name, column_name) -> column UUID

for table in tables:
    table_id_map[table.name] = table.id
    # get_database may return tables without column details (lightweight response)
    # fetch each table individually to guarantee columns are populated
    if not table.columns:
        table = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    print(f"  Table '{table.name}' -> {table.id} ({len(table.columns)} columns)")
    for col in table.columns:
        column_id_map[(table.name, col.name)] = col.id

print(f"\nMapped {len(table_id_map)} tables, {len(column_id_map)} columns")
if len(column_id_map) == 0:
    raise RuntimeError("No columns found — cannot upload. Check DBRepo schema.")

# Sanity check — print all discovered (table, column) pairs (fix point 4)
print("\nDiscovered columns in DBRepo:")
for k in sorted(column_id_map.keys()):
    print(f"  {k[0]}.{k[1]}")


  Table 'weather_measurement' -> 2212bed4-ef8f-4d95-bb65-20b2adb28abd (26 columns)
  Table 'time_dimension' -> 9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2 (3 columns)
  Table 'station' -> e6779029-ce40-4a9a-ad17-147e183dc757 (8 columns)

Mapped 3 tables, 37 columns

Discovered columns in DBRepo:
  station.altitude_m
  station.district_code
  station.latitude_deg
  station.longitude_deg
  station.nuts_code
  station.station_name
  station.station_num
  station.sub_district_code
  time_dimension.ref_month
  time_dimension.ref_year
  time_dimension.time_id
  weather_measurement.mean_t_max_c
  weather_measurement.mean_t_min_c
  weather_measurement.measurement_id
  weather_measurement.num_clear
  weather_measurement.num_cloud
  weather_measurement.num_frost
  weather_measurement.num_heat
  weather_measurement.num_ice
  weather_measurement.num_precp_01
  weather_measurement.num_summer
  weather_measurement.num_wind_vel60
  weather_measurement.p_max_hpa
  weather_measurement.p_mean_hpa
  weather_mea

## Step 4 — Load the semantic mapping CSV

In [7]:
import csv

mappings = []
with open(MAPPING_FILE, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        mappings.append(row)

print(f"Loaded {len(mappings)} mappings from CSV")
for m in mappings[:3]:
    print(m)


Loaded 37 mappings from CSV
{'table_name': 'weather_measurement', 'column_name': 'measurement_id', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Measurement identifier'}
{'table_name': 'weather_measurement', 'column_name': 'station_num', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Station identifier'}
{'table_name': 'weather_measurement', 'column_name': 'time_id', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Time identifier'}


## Step 5 — Upload each semantic concept using the official dbrepo client

Uses `client.update_table_column(database_id, table_id, column_id, concept_uri=uri)`.

> **Note on `ontology_label`:** Only `concept_uri` is sent to DBRepo. The label field in the CSV 
is for human readability. DBRepo resolves the concept name internally from its own concept 
store using the URI — no separate label call is needed.

In [8]:
success, skipped, errors = 0, 0, []

for row in mappings:
    tname = row["table_name"]
    cname = row["column_name"]
    uri   = row["ontology_uri"]
    # ontology_label is human-readable only — DBRepo derives label from URI via its concept store

    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))

    if tid is None or cid is None:
        skipped += 1
        print(f"  SKIP  {tname}.{cname} — not found in DBRepo schema")
        continue

    try:
        # Verified signature: update_table_column(database_id, table_id, column_id,
        #                                         concept_uri=None, unit_uri=None)
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=tid,
            column_id=cid,
            concept_uri=uri
        )
        success += 1
        print(f"  OK    {tname}.{cname} -> {uri}")
    except Exception as e:
        errors.append((tname, cname, str(e)))
        print(f"  FAIL  {tname}.{cname} -> {e}")

print()
print(f"Result: {success} uploaded, {skipped} skipped, {len(errors)} failed")
if errors:
    print("\nFailed rows:")
    for e in errors:
        print(f"  {e[0]}.{e[1]}: {e[2]}")


  OK    weather_measurement.measurement_id -> http://purl.org/dc/terms/identifier
  OK    weather_measurement.station_num -> http://purl.org/dc/terms/identifier
  OK    weather_measurement.time_id -> http://purl.org/dc/terms/identifier
  OK    weather_measurement.t_mean_c -> http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.t_max_c -> http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.t_min_c -> http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.mean_t_max_c -> http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.mean_t_min_c -> http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.p_mean_hpa -> http://qudt.org/vocab/quantitykind/Pressure
  OK    weather_measurement.p_max_hpa -> http://qudt.org/vocab/quantitykind/Pressure
  OK    weather_measurement.p_min_hpa -> http://qudt.org/vocab/quantitykind/Pressure
  OK    weather_measurement.precp_sum_mm -> http://qudt.org/vocab/q

## Step 6 — Verify: read back spot-checks from DBRepo

In [9]:
spot_checks = [
    ("weather_measurement", "t_mean_c"),
    ("weather_measurement", "precp_sum_mm"),
    ("station",             "nuts_code"),
    ("station",             "latitude_deg"),
    ("time_dimension",      "ref_year"),
]

print("Spot-check verification:")
all_ok = True
for tname, cname in spot_checks:
    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))
    if tid is None or cid is None:
        print(f"  SKIP  {tname}.{cname} — ID not found")
        continue
    try:
        tbl = client.get_table(database_id=DATABASE_ID, table_id=tid)  # returns Table object
        matched = next((c for c in tbl.columns if c.name == cname), None)
        if matched:
            print(f"  OK    {tname}.{cname}")
            print(f"        concept_uri = {matched.concept_uri}")
            if not matched.concept_uri:
                print(f"        WARNING: concept_uri is empty — upload may have failed")
                all_ok = False
        else:
            print(f"  MISS  {tname}.{cname} — column not in response")
            all_ok = False
    except Exception as e:
        print(f"  FAIL  {tname}.{cname} -> {e}")
        all_ok = False

print()
print("All spot-checks passed" if all_ok else "Some checks failed — review above")


Spot-check verification:
  OK    weather_measurement.t_mean_c
        concept_uri = http://qudt.org/vocab/quantitykind/Temperature
  OK    weather_measurement.precp_sum_mm
        concept_uri = http://qudt.org/vocab/quantitykind/LengthOrDepth
  OK    station.nuts_code
        concept_uri = http://data.europa.eu/nuts/code
  OK    station.latitude_deg
        concept_uri = http://www.w3.org/2003/01/geo/wgs84_pos#lat
  OK    time_dimension.ref_year
        concept_uri = http://www.w3.org/2006/time#year

All spot-checks passed
